## Notebook to try out my idea to create embeddings for my dashboard for task 6 using plotly

In [0]:
%python
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Läsa in data ifrån mina två growth marts i transformations/gold
df_official = spark.sql(
    """
    SELECT 
        SUM(total_events_hosted) AS total_events,
        SUM(total_global_finishers) AS total_finishers
    FROM marathos.gold.mart_event_growth_official"""
).toPandas()

df_verified = spark.sql(
    """
    SELECT 
        SUM(total_global_finishers) AS verified_finishers
    FROM marathos.gold.mart_event_growth_verified"""
).toPandas()

- Why use `.toPandas()` in the cell above?
    - Reason is because im aggregating it down to ONE row which is trivial for pandas. It WOULD start to become a problem if I would try to aggregate 6.8mil rows with pandas.

In [0]:
total_events = int(df_official["total_events"].iloc[0])
total_finishers = int(df_official["total_finishers"].iloc[0])

verified = int(df_verified["verified_finishers"].iloc[0])
no_show_rate = round((total_finishers - verified) / total_finishers * 100, 1)

print(f"Events: {total_events:,}") # :, för att separera siffrorna för läsbarhet
print(f"Official: {total_finishers:,}")
print(f"Verified: {verified:,}")
print(f"No shows: {no_show_rate}%")

- Using `:,` in an f-string is a format specifier. Python automatically formats big numbers with thousand separators. 

In [0]:
%python
# Byggandet av mina KPI kort
fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])

fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_events,
        title={"text": "Total Events<br><span style='font-size:0.8em'>All time</span>"},
        number={"valueformat": ","},
    ),
    row=1,
    col=1,
)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_finishers,
    title={"text": "Official Finishers<br><span style='font-size:0.8em'>Reported by organizers</span>"},
    number={"valueformat": ","}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=verified,
    title={"text": "Verified Results<br><span style='font-size:0.8em'>In our database</span>"},
    number={"valueformat": ","}
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode="number",
    value=no_show_rate,
    title={"text": "No-Show Rate<br><span style='font-size:0.8em'>Official vs Verified</span>"},
    number={"suffix": "%", "valueformat": ".2f"}
), row=1, col=4)

fig.update_layout(
    title_text="Marathos Global Overview KPIs",
    template="plotly_dark", # Vilken färg template som används
    height=250,
    margin=dict(t=85, b=20, l=20, r=20)
)

fig.show()

## The difference between plotting in my bronze EDA and here

- In my bronze EDA i used `plotly.express` (px). It is fast, easy and only takes one call.
    - Example: Using `px.bar()`, `px.line()` or `px.scatter()`. 
    - Strenghts: It is super effective to visualise data and using it for EDA.

- In this notebook I took advantage of and used `plotly.graph_objects` (go). It is more complex but gives the develpoer FULL control of everything.
    - Example: Using `go.Indicator()`, `go.Figure()`, or `make_subplots()`
    - Strenghts: It is very good for dashboards or layouts that needs adapting.

- `make_subplots` lets me place several graphs into a grid, which is shown in cell above with `rows = 1, cols = 4` and what that means is that the grid is made up of one row with 4 columns.
- `specs` tells plotly that each cell is of the type `indicator`, which in itself is a special format for KPI numbers.

##  Now my KPI cards are created and tested. Time to move on to my growth over time marts.

In [0]:
df_growth = spark.sql("""
    SELECT 
        o.year_of_event,
        o.total_events_hosted,
        o.total_global_finishers  AS official_signups,
        v.total_global_finishers  AS verified_finishers
    FROM marathos.gold.mart_event_growth_official o
    JOIN marathos.gold.mart_event_growth_verified v
        ON o.year_of_event = v.year_of_event
    ORDER BY o.year_of_event ASC
""").toPandas()

df_growth.head(10)

In [0]:
%python
# Checking the shape of my DF
df_growth.shape

In [0]:
%python

# Checking how the data looks over time to figure out what I should plot.
print(df_growth.head(5))
print("---")
print(df_growth.tail(5))
print("---")
print(df_growth[df_growth["year_of_event"] >= 1990].shape)

- With the cells above relating to growth over time there is two ways I could go.
    - Option 1; Show ALL data 1807 - 2022 that entails 180 years of almost nothing, then a massive spike.
        - This is hard to read but historically honest.

    - Option 2; Show data from 1990-2022. This skips the 180 years of almost nothing and focuses on growth, covid dip(relatable for us all).
        - This is more readable but I cut out the historical value.

- For this purpose, I aim to focus on option 2. Focus on GROWTH and then perhaps if there is time i will add "historical data"

In [0]:
%python
# Filtering on 1990+ for growth graph


# Using .copy() to avoid pandas SettingWithCopyWarning issues later on.
df_modern = df_growth[df_growth["year_of_event"] >= 1990].copy()

- Building my growth graph with double y axis
    - Left y axis represents nr of events
    - Right y axis represents nr of finishers
    - Lines represent: Number of events hosted, number of verified finishers and number of official finishers of races.


In [0]:
%python
# Building my growth over time graphs focusing on option 2 from text cell above.

fig = make_subplots(specs = [[{"secondary_y": True}]])

# Vänster sida av y axis = antal events
fig.add_trace(
    go.Scatter(
        x = df_modern["year_of_event"],
        y = df_modern["total_events_hosted"],
        name = "Events hosted",
        mode = "lines+markers",
        line = dict(color = "#00CC96", width = 2)
    ),
    secondary_y = False
)

# Höger sida av y axis = antal finishers
fig.add_trace(
    go.Scatter(
        x = df_modern["year_of_event"],
        y = df_modern["official_signups"],
        name = "Official Signups",
        mode = "lines+markers",
        line = dict(color = "#EF553B", width = 2)
    ),
    secondary_y = True
)

fig.add_trace(
    go.Scatter(
        x=df_modern["year_of_event"],
        y=df_modern["verified_finishers"],
        name="Verified finishers",
        mode="lines+markers",
        line=dict(color="#636EFA", width=2, dash="dot")
    ),
    secondary_y=True
)

- Building the layout for my growth graph and axis titles


In [0]:
%python
 # Layout and axis titles for my growth graph
fig.update_layout(
    title_text="Marathos — Global Growth 1990–2022",
    template="plotly_dark",
    height=450,
    hovermode="x unified"   #  Visar alla värden vid hover över på en punkt
)

fig.update_yaxes(title_text="Events hosted", secondary_y=False)
fig.update_yaxes(title_text="Finishers", secondary_y=True)
fig.update_xaxes(title_text="Year")

fig.show()

### Building graphs for my seasonal analysis using mart_seasonal_events_country.sql script


In [0]:
%python

df_seasonal = spark.sql("""
    SELECT 
        event_country,
        month,
        total_events,
        total_finishers,
        ROUND(avg_performance, 2) AS avg_performance
    FROM marathos.gold.mart_seasonal_events_country
    ORDER BY month ASC
""").toPandas()

# Joina in landnamn från dim_country ===
df_countries = spark.sql("""
    SELECT iso_code, country_name
    FROM marathos.gold.dim_country
""").toPandas()

df_seasonal = df_seasonal.merge(
    df_countries,
    left_on="event_country",
    right_on="iso_code",
    how="left"
)

In [0]:
%python 

# Verifying shape and data in df_seasonal
print(df_seasonal.shape)
print(df_seasonal["event_country"].nunique())
print(df_seasonal.head(5))

# Sanity check vilka länder saknar mappning?
missing = df_seasonal[df_seasonal["country_name"].isna()]["event_country"].unique()
print(f"Nations without mapping: {missing}")

- Data is verified, but I will not use avg_performance here, it will skew the data since im getting ah `h-event` where `performance = km` should be. 


In [0]:
%python

# Top 50 nations
top_countries = (
    df_seasonal.groupby("country_name")["total_events"]
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index.tolist()
)

df_top = df_seasonal[df_seasonal["country_name"].isin(top_countries)]

- Time to start building grouped bar chart with dropdown menue

In [0]:
%python

fig = go.Figure()

for country in top_countries:
    df_country = df_top[df_top["country_name"] == country]
    
    fig.add_trace(go.Bar(
        x=df_country["month"],
        y=df_country["total_events"],
        name=country,
        visible=(country == top_countries[0])
    ))

buttons = []
for i, country in enumerate(top_countries):
    visible = [j == i for j in range(len(top_countries))]
    buttons.append(dict(
        label=country,
        method="update",
        args=[{"visible": visible}]
    ))

fig.update_layout(
    title_text="Total Number of Events in Dataset - Top 50 Nations",
    template="plotly_dark",
    height=450,
    xaxis=dict(
        title="Month",
        tickmode="array",
        tickvals=list(range(1, 13)),
        ticktext=["Jan","Feb","Mar","Apr","May","Jun",
                  "Jul","Aug","Sep","Oct","Nov","Dec"]
    ),
    yaxis_title="Total Events",
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.0,
        y=1.15,
        xanchor="right"
    )]
)

fig.show()

### 